# E2 Latent Steering — Whisper Decoder
Position-based and DTW-α-timeline interpolation evaluated with standard Whisper-small.
Two figures per method: (1) overall WER | ΔWER, (2) per-L1 6×2 grid.

In [1]:
import torch, json, re, sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import jiwer
from pathlib import Path
from tqdm.notebook import tqdm
from dtaidistance import dtw_ndim
from transformers import WhisperForConditionalGeneration, WhisperProcessor
from transformers.modeling_outputs import BaseModelOutput

ROOT = Path("/rds/general/user/tsv22/home/accent-robust-asr")
sys.path.insert(0, str(ROOT))
from src.config import RANDOM_SEED

sns.set_style("whitegrid")
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")
if device == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")

Device: cuda
GPU: Quadro RTX 6000


In [2]:
def load_checkpoint(path: str) -> np.ndarray:
    ckpt = torch.load(path, map_location="cpu", weights_only=False)
    hs = ckpt["hidden_states"]
    return hs.to(torch.float32).cpu().numpy() if isinstance(hs, torch.Tensor) else np.array(hs, dtype=np.float32)

def norm(s: str) -> str:
    s = s.lower().strip()
    s = re.sub(r"[^\w\s]", "", s)
    return re.sub(r"\s+", " ", s).strip()

def build_reference_lookup(wer_csv: str) -> dict:
    df = pd.read_csv(wer_csv)
    prompt_col  = next(c for c in ["utterance_id", "prompt_id", "utt_id"] if c in df.columns)
    speaker_col = next(c for c in ["speaker", "spk"]                       if c in df.columns)
    ref_col     = next(c for c in ["reference_norm", "reference", "text"]  if c in df.columns)
    ref_data = {}
    for _, row in df.iterrows():
        val = str(row[ref_col]).strip()
        if pd.notna(val) and val.lower() != "nan":
            ref_data[(str(row[prompt_col]), str(row[speaker_col]))] = val
    return ref_data

print("\u2713 Utility functions defined")

✓ Utility functions defined


In [9]:
mapping_cache = ROOT / "src/analysis/cache/utterance_mapping.json"
wer_csv       = ROOT / "results/model_perf_comparison/whisfusion_finetuned_predictions.csv"
output_dir    = ROOT / "src/analysis/results/e2_steering_whisper"
output_dir.mkdir(parents=True, exist_ok=True)

with open(mapping_cache) as f:
    mapping = json.load(f)
ref_data = build_reference_lookup(str(wer_csv))
ref_data = {k: v for k, v in ref_data.items() if pd.notna(v) and v.lower() != "nan"}

# Keep only prompts with English + at least one valid L2
mapping = {
    pid: {l1: info for l1, info in l1d.items() if (pid, info["speaker"]) in ref_data}
    for pid, l1d in mapping.items()
}
mapping = {pid: d for pid, d in mapping.items() if "English" in d and len(d) > 1}
print(f"Prompts: {len(mapping)}  |  References: {len(ref_data)}")

Prompts: 1131  |  References: 7795


In [10]:
NUM_PER_L1 = 100  # utterances per L1; None = full dataset
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)

if NUM_PER_L1 is not None:
    # Stratified: sample NUM_PER_L1 prompts per L1, union the results
    selected = set()
    l1_counts = {}
    for l1 in sorted({l1 for l1d in mapping.values() for l1 in l1d if l1 != "English"}):
        candidates = [pid for pid, l1d in mapping.items() if l1 in l1d]
        chosen = np.random.choice(candidates, size=min(NUM_PER_L1, len(candidates)), replace=False)
        selected.update(chosen)
        l1_counts[l1] = len(chosen)
    mapping = {pid: mapping[pid] for pid in selected}
    print(f"Stratified subsample: {len(mapping)} prompts total")
    for l1, n in l1_counts.items():
        print(f"  {l1}: {n} utterances")
else:
    print(f"Full dataset: {len(mapping)} prompts")

Stratified subsample: 486 prompts total
  Arabic: 100 utterances
  Chinese: 100 utterances
  Hindi: 100 utterances
  Korean: 100 utterances
  Spanish: 100 utterances
  Vietnamese: 100 utterances


In [11]:
print("Loading Whisper-small...")
_cache        = str(ROOT / ".cache")
whisper_model = WhisperForConditionalGeneration.from_pretrained(
    "openai/whisper-small", cache_dir=_cache).eval().to(device)
whisper_proc  = WhisperProcessor.from_pretrained(
    "openai/whisper-small", cache_dir=_cache)
print(f"  \u2713 Loaded on {device}")

@torch.inference_mode()
def whisper_decode(hidden_state_np: np.ndarray) -> str:
    enc_out = BaseModelOutput(
        last_hidden_state=torch.from_numpy(hidden_state_np).unsqueeze(0).to(device)
    )
    ids = whisper_model.generate(
        encoder_outputs=enc_out, language="en", task="transcribe", temperature=0.0
    )
    return whisper_proc.batch_decode(ids, skip_special_tokens=True)[0].strip()

# Sanity-check decode
_path = next(info["path"] for l1d in mapping.values()
             for info in l1d.values() if Path(info["path"]).exists())
print(f"  Decode test: '{whisper_decode(load_checkpoint(_path))[:70]}'")

Loading Whisper-small...


Loading weights:   0%|          | 0/479 [00:00<?, ?it/s]

  ✓ Loaded on cuda
  Decode test: 'It was a superb picture.'


In [12]:
def pad_to_1500(steered, l2_full, l2_end, eng_full, eng_end, alpha):
    """Pad steered speech to 1500 frames, blending silence from both speakers.
    Exact endpoints: alpha=0 -> l2_full[:1500], alpha=1 -> eng_full[:1500].
    """
    N = len(steered)
    if N >= 1500:
        return steered[:1500].astype(np.float32)
    need = 1500 - N

    def _fit(arr, n):
        return arr[:n] if len(arr) >= n else np.vstack([arr, np.tile(arr[-1:], (n - len(arr), 1))])

    sil = (1 - alpha) * _fit(l2_full[l2_end:], need) + alpha * _fit(eng_full[eng_end:], need)
    return np.vstack([steered, sil]).astype(np.float32)

print("\u2713 DTW / padding helpers defined")

✓ DTW / padding helpers defined


In [15]:
alpha_values   = [0.0, 0.25, 0.5, 0.75, 0.9, 1.0]
pos_cache_path = output_dir / "whisper_position_steering.csv"

if pos_cache_path.exists():
    print(f"Loading cached results from {pos_cache_path}...")
    pos_df = pd.read_csv(pos_cache_path)
    print(f"  \u2713 {len(pos_df)} rows")
else:
    print("Running position-based steering...")
    rows = []
    with tqdm(total=len(mapping) * 6 * len(alpha_values), unit="decode") as pbar:
        for prompt_id, l1d in mapping.items():
            if "English" not in l1d:
                pbar.update(6 * len(alpha_values)); continue
            try:
                eng_full = load_checkpoint(l1d["English"]["path"])
            except Exception:
                pbar.update(6 * len(alpha_values)); continue

            for l1, info in l1d.items():
                if l1 == "English":
                    continue
                ref = ref_data.get((prompt_id, info["speaker"]))
                if ref is None:
                    pbar.update(len(alpha_values)); continue
                try:
                    l2_full = load_checkpoint(info["path"])
                except Exception:
                    pbar.update(len(alpha_values)); continue

                for alpha in alpha_values:
                    try:
                        steered = ((1 - alpha) * l2_full + alpha * eng_full).astype(np.float32)
                        pred    = whisper_decode(steered)
                        rows.append({"prompt_id": prompt_id, "L1": l1,
                                     "speaker": info["speaker"], "alpha": alpha,
                                     "wer": jiwer.wer(norm(ref), norm(pred))})
                    except Exception:
                        pass
                    pbar.update(1)

    pos_df = pd.DataFrame(rows)
    pos_df.to_csv(pos_cache_path, index=False)
    print(f"\u2713 Saved {len(pos_df)} rows to {pos_cache_path}")

print("\nWER by alpha (position-based):")
print(pos_df.groupby("alpha")["wer"].agg(["mean","std","min","max"]).round(4))

Running position-based steering...


  0%|          | 0/17496 [00:00<?, ?decode/s]

KeyboardInterrupt: 

In [ ]:
dtw_cache_path = output_dir / "whisper_dtw_alpha_timeline_steering.csv"

if dtw_cache_path.exists():
    print(f"Loading cached results from {dtw_cache_path}...")
    dtw_df = pd.read_csv(dtw_cache_path)
    print(f"  \u2713 {len(dtw_df)} rows")
else:
    print("Running DTW-\u03b1-timeline steering...")
    rows = []
    with tqdm(total=len(mapping) * 6 * len(alpha_values), unit="decode") as pbar:
        for prompt_id, l1d in mapping.items():
            if "English" not in l1d:
                pbar.update(6 * len(alpha_values)); continue
            eng_info = l1d["English"]
            eng_end  = eng_info.get("speech_end_frame")
            if not eng_end or not Path(eng_info["path"]).exists():
                pbar.update(6 * len(alpha_values)); continue
            try:
                eng_full  = load_checkpoint(eng_info["path"])
                eng_state = eng_full[:eng_end]
            except Exception:
                pbar.update(6 * len(alpha_values)); continue

            for l1, info in l1d.items():
                if l1 == "English":
                    continue
                l2_end = info.get("speech_end_frame")
                ref    = ref_data.get((prompt_id, info["speaker"]))
                if not l2_end or ref is None or not Path(info["path"]).exists():
                    pbar.update(len(alpha_values)); continue
                try:
                    l2_full  = load_checkpoint(info["path"])
                    l2_state = l2_full[:l2_end]
                except Exception:
                    pbar.update(len(alpha_values)); continue

                # Compute DTW path once per pair
                path_arr = np.array(dtw_ndim.warping_path(eng_state, l2_state))
                T_eng, T_l2 = len(eng_state), len(l2_state)
                i_norm = path_arr[:, 0] / max(T_eng - 1, 1)
                j_norm = path_arr[:, 1] / max(T_l2  - 1, 1)

                for alpha in alpha_values:
                    try:
                        N     = max(1, round((1 - alpha) * T_l2 + alpha * T_eng))
                        t_k   = (1 - alpha) * j_norm + alpha * i_norm
                        out_t = np.linspace(0.0, 1.0, N)
                        idx_r = np.clip(np.searchsorted(t_k, out_t), 0, len(t_k) - 1)
                        idx_l = np.clip(idx_r - 1, 0, len(t_k) - 1)
                        k_idx = np.where(np.abs(t_k[idx_l] - out_t) <= np.abs(t_k[idx_r] - out_t),
                                         idx_l, idx_r)
                        steered = ((1 - alpha) * l2_state[path_arr[k_idx, 1]]
                                   + alpha     * eng_state[path_arr[k_idx, 0]]).astype(np.float32)
                        padded  = pad_to_1500(steered, l2_full, l2_end, eng_full, eng_end, alpha)
                        pred    = whisper_decode(padded)
                        rows.append({"prompt_id": prompt_id, "L1": l1,
                                     "speaker": info["speaker"], "alpha": alpha,
                                     "wer": jiwer.wer(norm(ref), norm(pred))})
                    except Exception:
                        pass
                    pbar.update(1)

    dtw_df = pd.DataFrame(rows)
    dtw_df.to_csv(dtw_cache_path, index=False)
    print(f"\u2713 Saved {len(dtw_df)} rows to {dtw_cache_path}")

print("\nWER by alpha (DTW-\u03b1-timeline):")
print(dtw_df.groupby("alpha")["wer"].agg(["mean","std","min","max"]).round(4))

In [ ]:
def plot_overall(df, title, save_path=None):
    """1×2 figure: WER and ΔWER vs alpha for the full dataset."""
    stats = (df.groupby("alpha")["wer"]
               .agg(mean="mean", sem=lambda x: x.std() / len(x)**0.5)
               .reset_index().sort_values("alpha"))
    alphas   = stats["alpha"].values
    wers     = stats["mean"].values
    sems     = stats["sem"].values
    baseline = stats.loc[stats["alpha"] == 0.0, "mean"].values[0]
    deltas   = wers - baseline
    best_i   = int(np.argmin(wers))

    fig, (ax_w, ax_d) = plt.subplots(1, 2, figsize=(13, 4))

    # WER
    ax_w.plot(alphas, wers, "o-", lw=2, ms=6, color="steelblue")
    ax_w.fill_between(alphas, wers - sems, wers + sems, alpha=0.18, color="steelblue")
    ax_w.axhline(baseline, color="dimgray", ls="--", lw=1.5, label=f"L2 baseline  {baseline:.3f}")
    ax_w.plot(alphas[best_i], wers[best_i], "*", ms=15, color="gold",
              mec="darkorange", zorder=5, label=f"Best α={alphas[best_i]}  {wers[best_i]:.3f}")
    ax_w.set_xlabel("α  (0 = pure L2,  1 = pure English)"); ax_w.set_ylabel("WER")
    ax_w.set_title(f"{title} — WER", fontsize=12, fontweight="bold")
    ax_w.legend(fontsize=9); ax_w.grid(True, alpha=0.3)

    # ΔWER
    bar_colors = ["#e74c3c" if d > 0 else "#27ae60" for d in deltas]
    ax_d.bar(alphas, deltas, color=bar_colors, alpha=0.75, width=0.05, zorder=2)
    ax_d.plot(alphas, deltas, "o-", lw=1.5, ms=5, color="steelblue", zorder=3)
    ax_d.axhline(0, color="dimgray", ls="--", lw=1.5)
    ax_d.plot(alphas[best_i], deltas[best_i], "*", ms=15, color="gold",
              mec="darkorange", zorder=5)
    ax_d.set_xlabel("α"); ax_d.set_ylabel("ΔWER  (vs L2 baseline)")
    ax_d.set_title(f"{title} — ΔWER", fontsize=12, fontweight="bold")
    ax_d.grid(True, alpha=0.3)

    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches="tight")
        print(f"\u2713 Saved {save_path.name}")
    plt.show()


def plot_per_l1(df, title, save_path=None):
    """6×2 grid: one row per L1, WER | ΔWER columns."""
    l1s = sorted(df["L1"].unique())
    fig, axes = plt.subplots(len(l1s), 2, figsize=(13, 3.5 * len(l1s)))

    for row_i, l1 in enumerate(l1s):
        sub    = df[df["L1"] == l1]
        stats  = (sub.groupby("alpha")["wer"]
                     .agg(mean="mean", sem=lambda x: x.std() / len(x)**0.5)
                     .reset_index().sort_values("alpha"))
        alphas   = stats["alpha"].values
        wers     = stats["mean"].values
        sems     = stats["sem"].values
        baseline = stats.loc[stats["alpha"] == 0.0, "mean"].values[0]
        deltas   = wers - baseline
        best_i   = int(np.argmin(wers))

        ax_w, ax_d = axes[row_i, 0], axes[row_i, 1]

        ax_w.plot(alphas, wers, "o-", lw=2, ms=5, color="steelblue")
        ax_w.fill_between(alphas, wers - sems, wers + sems, alpha=0.18, color="steelblue")
        ax_w.axhline(baseline, color="dimgray", ls="--", lw=1.5)
        ax_w.plot(alphas[best_i], wers[best_i], "*", ms=13, color="gold",
                  mec="darkorange", zorder=5, label=f"Best α={alphas[best_i]}")
        ax_w.set_title(f"{l1}", fontsize=10, fontweight="bold")
        ax_w.set_ylabel("WER", fontsize=9); ax_w.set_xlabel("α", fontsize=9)
        ax_w.legend(fontsize=8); ax_w.grid(True, alpha=0.3)

        bar_colors = ["#e74c3c" if d > 0 else "#27ae60" for d in deltas]
        ax_d.bar(alphas, deltas, color=bar_colors, alpha=0.75, width=0.05, zorder=2)
        ax_d.plot(alphas, deltas, "o-", lw=1.5, ms=4, color="steelblue", zorder=3)
        ax_d.axhline(0, color="dimgray", ls="--", lw=1.5)
        ax_d.plot(alphas[best_i], deltas[best_i], "*", ms=13, color="gold",
                  mec="darkorange", zorder=5)
        ax_d.set_title(f"{l1} (Δ)", fontsize=10, fontweight="bold")
        ax_d.set_ylabel("ΔWER", fontsize=9); ax_d.set_xlabel("α", fontsize=9)
        ax_d.grid(True, alpha=0.3)

    fig.suptitle(f"{title} — per L1", fontsize=12, fontweight="bold", y=1.01)
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches="tight")
        print(f"\u2713 Saved {save_path.name}")
    plt.show()

print("\u2713 Plot helpers defined")

## Position-Based Steering

In [ ]:
plot_overall(pos_df, "Position-Based",
             save_path=output_dir / "whisper_position_overall.png")

In [ ]:
plot_per_l1(pos_df, "Position-Based",
            save_path=output_dir / "whisper_position_per_l1.png")

## DTW-α-Timeline Steering

In [ ]:
plot_overall(dtw_df, "DTW-\u03b1-Timeline",
             save_path=output_dir / "whisper_dtw_alpha_overall.png")

In [ ]:
plot_per_l1(dtw_df, "DTW-\u03b1-Timeline",
            save_path=output_dir / "whisper_dtw_alpha_per_l1.png")